# 🌐 Serveur API FastAPI pour le Chatbot Fine-Tuné (DGRE)
Ce notebook charge votre modèle fine-tuné et expose une **API REST publique via un tunnel URL (Ngrok ou LocalTunnel)**.
Vous pourrez ainsi envoyer des requêtes HTTP POST depuis votre application locale (Streamlit, React, Python, etc.).

### Étape 1 : Installation des dépendances pour le serveur API

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" trl peft accelerate bitsandbytes
!pip install fastapi uvicorn pydantic pyngrok nest_asyncio

### Étape 2 : Importer vos poids fine-tunés (`lora_model_dgre.zip`)
Si vous venez d'exécuter le notebook d'entraînement dans la même session, le dossier `lora_model_dgre` existe déjà.
Sinon, exécutez cette cellule pour importer votre fichier `.zip`.

In [ ]:
import os
from google.colab import files

if not os.path.exists('lora_model_dgre'):
    if os.path.exists('lora_model_dgre.zip'):
        !unzip -q lora_model_dgre.zip
    else:
        print("Veuillez uploader votre archive lora_model_dgre.zip :")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.zip'):
                !unzip -q "$fn"
                break
print("✅ Dossier du modèle LoRA présent !")

### Étape 3 : Chargement du Modèle en mémoire GPU

In [ ]:
from unsloth import FastLanguageModel
import torch

model_path = "lora_model_dgre" if os.path.exists("lora_model_dgre") else "Qwen/Qwen2.5-Coder-7B-Instruct"
max_seq_length = 2048

print(f"⏳ Chargement du modèle depuis {model_path}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)
print("✅ Modèle prêt pour l'inférence !")

### Étape 4 : Définition de l'API FastAPI

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Dict

app = FastAPI(title="DGRE Chatbot API", description="API de génération de requêtes SQL et code Python SIG")

class Message(BaseModel):
    role: str
    content: str

class QueryRequest(BaseModel):
    messages: List[Message]
    max_tokens: Optional[int] = 1024
    temperature: Optional[float] = 0.1

class QueryResponse(BaseModel):
    response: str

@app.get("/")
def root():
    return {"status": "ok", "model": "Qwen2.5-Coder-7B-Instruct-DGRE"}

@app.post("/api/chat", response_model=QueryResponse)
def chat_endpoint(req: QueryRequest):
    try:
        chat_list = [{"role": m.role, "content": m.content} for m in req.messages]
        
        inputs = tokenizer.apply_chat_template(
            chat_list,
            tokenize = True,
            add_generation_prompt = True,
            return_tensors = "pt"
        ).to("cuda")
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids = inputs,
                max_new_tokens = req.max_tokens,
                temperature = req.temperature,
                use_cache = True
            )
            
        reponse_texte = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
        return QueryResponse(response=reponse_texte)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

### Étape 5 : Création de l'URL publique avec Ngrok
1. Créez un compte gratuit sur [ngrok.com](https://ngrok.com) pour obtenir votre token d'authentification.
2. Collez votre token ci-dessous.

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn

# Insérez votre token Ngrok gratuit ici :
NGROK_AUTH_TOKEN = ""  # <-- METTRE VOTRE TOKEN NGROK ICI SI DISPONIBLE

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Ouvrir le tunnel HTTP sur le port 8000
public_url = ngrok.connect(8000)
print("=" * 65)
print(f"🚀 VOTRE URL PUBLIQUE API EST : {public_url.public_url}")
print(f"👉 Endpoint de chat : {public_url.public_url}/api/chat")
print("=" * 65)

# Lancement du serveur uvicorn en arrière-plan
nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)

### Étape 6 : Exemple d'appel depuis votre application Python locale
Voici le code que vous pouvez insérer dans votre application cliente (Streamlit ou script local) pour interroger cette URL :

In [ ]:
import requests

# Remplacez par votre URL Ngrok affichée ci-dessus
API_URL = f"{public_url.public_url}/api/chat"

payload = {
    "messages": [
        {"role": "system", "content": "Tu es un traducteur de langage naturel vers PostgreSQL/PostGIS expert pour la Tunisie (DGRE). Renvoie UNIQUEMENT un objet JSON valide avec 'sql_query' et 'target_col'."},
        {"role": "user", "content": "Donne moi les pluies pour Tataouine en octobre 2021"}
    ],
    "temperature": 0.1,
    "max_tokens": 512
}

res = requests.post(API_URL, json=payload)
print("Réponse de l'API :", res.json()["response"])